# Practice: Python classes, step by step

This notebook is a set of building blocks for working with classes in Python. Every step adds **one** idea to the step before, so you can go through it from top to bottom and run each cell.

The example is the one you meet in the [supermarket challenge](../supermarket-challenge/README.md): *HTW Markt*, a small shop on campus whose checkout counter runs a Python program. Here we build simplified versions of its products and its shopping cart ourselves, so that the challenge code looks familiar when you open it. This notebook does not solve any task of the challenge.

After this notebook you can

- write a class, create objects from it and call their methods;
- document a class with docstrings and type hints;
- protect an attribute with a check, and use a property;
- build one class out of others (composition) and a new class from an existing one (inheritance).

## Overview

| Part | What you practise |
|---|---|
| 1 · A class is a blueprint | `class`, objects, `__init__`, `self`, attributes |
| 2 · Methods | functions that belong to a class, calling them, arguments and default values |
| 3 · Printing an object | `__repr__` and `__str__` |
| 4 · Docstrings and type hints | documenting a class, `help()` |
| 5 · Many objects from one blueprint | lists and dictionaries of objects, class attributes |
| 6 · Protecting the data | checking values, raising `ValueError`, properties |
| 7 · A class that uses other classes | composition: a shopping cart |
| 8 · Inheritance | subclasses, overriding, `super()`, polymorphism, abstract classes |

```mermaid
flowchart LR
    A["1 · blueprint"] --> B["2 · methods"] --> C["3 · printing"] --> D["4 · docstrings"]
    D --> E["5 · many objects"] --> F["6 · protecting data"] --> G["7 · composition"] --> H["8 · inheritance"]
```

## 1 · A class is a blueprint

### Why not just a dictionary?

A product has a name and a price. We could store it in a dictionary. Run the next cell: it works, but a small typo in a key is not noticed.

In [ ]:
bread = {"name": "Bread", "net_price": 2.80}
milk = {"name": "Milk 1 l", "netprice": 1.09}   # typo: "netprice" instead of "net_price"

print(bread["net_price"])
print(milk.get("net_price"))                    # None: Python cannot know what we meant

A dictionary accepts any keys, so nothing tells us that every product must have a `name` and a `net_price`. A **class** describes this once: it is the **blueprint**, and every product made from it has the same attributes.

### The smallest class

`class Product:` starts the blueprint. `pass` means "nothing inside yet". Calling the class like a function, `Product()`, creates an **object** (also called an **instance**) from it.

In [ ]:
class Product:
    pass


bread = Product()      # one object made from the blueprint
milk = Product()       # a second, separate object

print(type(bread))
print(bread is milk)   # False: two different objects

### `__init__` and `self`: giving every object its data

`__init__` is a special method that Python runs **every time** an object is created. Its first parameter, `self`, is the new object itself. `self.name = name` stores the value on that object: an **attribute**.

**Predict.** After the next cell, what does `milk.net_price` show?

In [ ]:
class Product:
    def __init__(self, name, net_price):
        self.name = name                # an attribute of this one object
        self.net_price = net_price      # the price before VAT, in EUR


bread = Product("Bread", 2.80)
milk = Product("Milk 1 l", 1.09)

print(bread.name, bread.net_price)
print(milk.name, milk.net_price)

`Product("Bread", 2.80)` does three things: it creates an empty object, calls `__init__(new_object, "Bread", 2.80)`, and returns the object. We never pass `self` ourselves: Python does it.

Each object keeps its own values. Changing one does not change the other:

In [ ]:
bread.net_price = 2.90       # the baker raised the price

print(bread.net_price)       # 2.9
print(milk.net_price)        # still 1.09

> **Try it.** Create a third product, `cheese`, with the net price 3.49, and print its name.

In [ ]:
# your code here

## 2 · Methods: functions that belong to a class

A **method** is a function written inside a class. Its first parameter is again `self`, so the method can use the object's attributes.

The shelf price is the net price plus VAT. Food in Germany has 7 % VAT, so the factor is 1.07.

In [ ]:
class Product:
    def __init__(self, name, net_price):
        self.name = name
        self.net_price = net_price

    def gross_price(self):
        return round(self.net_price * 1.07, 2)   # net price plus 7 % VAT, rounded to cents


bread = Product("Bread", 2.80)
print(bread.gross_price())             # call the method: 2.80 * 1.07 = 2.996, so 3.0
print(bread.gross_price)               # without parentheses: the method itself, not its result
print(Product.gross_price(bread))      # the same call, written the long way: this is where self comes from

### Arguments and default values

A method can take more arguments, like any function. Not all products have 7 % VAT: soap has 19 %. We pass the rate as an argument, with **7 % as the default value** when we do not give one.

In [ ]:
class Product:
    def __init__(self, name, net_price):
        self.name = name
        self.net_price = net_price

    def gross_price(self, vat_rate=0.07):
        return round(self.net_price * (1 + vat_rate), 2)


bread = Product("Bread", 2.80)
dish_soap = Product("Dish soap", 1.95)

print(bread.gross_price())                   # default rate: 3.0
print(dish_soap.gross_price(0.19))           # positional argument: 2.32
print(dish_soap.gross_price(vat_rate=0.19))  # keyword argument: the same, easier to read

### A method can call another method

`price_for(quantity)` uses `gross_price()` through `self`. Writing the price rule once and reusing it means there is only one place to change it.

In [ ]:
class Product:
    def __init__(self, name, net_price):
        self.name = name
        self.net_price = net_price

    def gross_price(self, vat_rate=0.07):
        return round(self.net_price * (1 + vat_rate), 2)

    def price_for(self, quantity):
        return round(self.gross_price() * quantity, 2)


milk = Product("Milk 1 l", 1.09)
print(milk.gross_price())       # 1.17
print(milk.price_for(2))        # 2.34

## 3 · Printing an object: `__repr__` and `__str__`

**Predict.** What does `print(milk)` show for the class above?

In [ ]:
print(milk)

The default text only says the class and a memory address. Two special methods change it:

- `__repr__`: a precise text for developers, shown in the notebook and in lists. Ideally it looks like the code that creates the object.
- `__str__`: a friendly text for people, used by `print()` and `str()`. Without it, `print()` uses `__repr__`.

In [ ]:
class Product:
    def __init__(self, name, net_price):
        self.name = name
        self.net_price = net_price

    def gross_price(self, vat_rate=0.07):
        return round(self.net_price * (1 + vat_rate), 2)

    def __repr__(self):
        return f"Product({self.name!r}, {self.net_price})"

    def __str__(self):
        return f"{self.name}: {self.gross_price():.2f} EUR"


milk = Product("Milk 1 l", 1.09)
print(milk)              # __str__ : Milk 1 l: 1.17 EUR
print(repr(milk))        # __repr__: Product('Milk 1 l', 1.09)
[milk, Product("Bread", 2.80)]   # a list shows __repr__ of each object

## 4 · Docstrings and type hints

Code is read more often than it is written. A **docstring** is a text in triple quotes right after `class` or `def`: it says what the class or method is for. The challenge uses the *Google style*: a short first line, then sections such as `Args:`, `Returns:` and `Raises:`.

**Type hints** (`name: str`, `-> float`) say which kind of value is expected and returned. Python does not check them when the code runs; they help the reader, your editor and tools such as type checkers.

In [ ]:
class Product:
    """Something on the shelf, with a name and a net price (the price before VAT).

    Attributes:
        name: The name on the shelf label, for example "Bread".
        net_price: The price in EUR before VAT.
    """

    def __init__(self, name: str, net_price: float) -> None:
        self.name = name
        self.net_price = net_price

    def gross_price(self, vat_rate: float = 0.07) -> float:
        """The shelf price: the net price plus VAT, rounded to cents.

        Args:
            vat_rate: The VAT rate as a fraction, 0.07 for food (the default) or 0.19.

        Returns:
            The price in EUR that the customer pays.

        Example:
            Product("Bread", 2.80).gross_price() is 2.80 * 1.07 = 2.996, rounded 3.0.
        """
        return round(self.net_price * (1 + vat_rate), 2)

    def __repr__(self) -> str:
        return f"Product({self.name!r}, {self.net_price})"

Docstrings are not comments: Python keeps them. `help()` shows them, and so does your editor when you hover over a name.

In [ ]:
help(Product.gross_price)
print(Product.__doc__)                  # the docstring of the class
print(Product.gross_price.__annotations__)   # the type hints, as a dictionary

> **Good practice.** The first line of a docstring is one sentence that says what the thing *is* or *does*. A comment (`#`) explains *why* a line is written the way it is. Write docstrings for every class and every public method.

## 5 · Many objects from one blueprint

A shop has many products. Objects can go into a list like any other value, and a loop can call the same method on each of them.

In [ ]:
shelf = [
    Product("Apples 1 kg", 2.29),
    Product("Bread", 2.80),
    Product("Milk 1 l", 1.09),
    Product("Cheese", 3.49),
]

for product in shelf:
    print(product.name, product.gross_price())

**Predict.** Which product is the cheapest, and what is the shelf price of one of each?

In [ ]:
cheapest = min(shelf, key=lambda product: product.gross_price())   # compare by shelf price
total = round(sum(product.gross_price() for product in shelf), 2)

print("cheapest:", cheapest)
print("one of each:", total, "EUR")

The checkout finds a product by its name. A dictionary from name to object makes that one step, as the catalogue of the challenge does:

In [ ]:
product_by_name = {product.name: product for product in shelf}

print(product_by_name["Cheese"])
print(product_by_name["Cheese"].gross_price())

### Class attributes: one value for all objects

An attribute set with `self.` belongs to one object (an **instance attribute**). An attribute written directly in the class body belongs to the class and is **shared** by all its objects: a **class attribute**. The challenge uses one for the category of a product.

In [ ]:
class Product:
    category = "food"          # class attribute: the same for every Product

    def __init__(self, name: str, net_price: float) -> None:
        self.name = name       # instance attributes: different for every object
        self.net_price = net_price


bread = Product("Bread", 2.80)
milk = Product("Milk 1 l", 1.09)
print(bread.category, milk.category, Product.category)

## 6 · Protecting the data

**Predict.** Does Python stop a negative price?

In [ ]:
class Product:
    def __init__(self, name: str, net_price: float) -> None:
        self.name = name
        self.net_price = net_price


broken = Product("Bread", -2.80)
print(broken.net_price)        # accepted: the shop would now pay the customer

Nothing stops it. A class can **check its own data** and refuse wrong values by raising an exception. `raise ValueError(...)` stops the code with an error message that says what is wrong.

In [ ]:
class Product:
    def __init__(self, name: str, net_price: float) -> None:
        if net_price < 0:
            raise ValueError(f"the price of {name!r} cannot be negative")
        self.name = name
        self.net_price = net_price


try:
    Product("Bread", -2.80)
except ValueError as error:          # catch the error so the notebook keeps running
    print("refused:", error)

The check in `__init__` protects only the moment of creation: `bread.net_price = -1` would still be accepted later. A **property** runs a check every time the attribute is set. This is **encapsulation**: the value is stored in `_net_price` (the leading underscore means "internal, do not touch from outside"), and outside code reads and writes `net_price`, which runs the two methods below.

In [ ]:
class Product:
    """A product whose net price can never be negative."""

    def __init__(self, name: str, net_price: float) -> None:
        self.name = name
        self.net_price = net_price          # runs the setter below, so the check runs here too

    @property
    def net_price(self) -> float:
        """The price in EUR before VAT."""
        return self._net_price

    @net_price.setter
    def net_price(self, new_net_price: float) -> None:
        if new_net_price < 0:
            raise ValueError(f"the price of {self.name!r} cannot be negative")
        self._net_price = new_net_price


bread = Product("Bread", 2.80)
bread.net_price = 2.90                      # allowed: runs the setter
print(bread.net_price)

try:
    bread.net_price = -1                    # refused, also after creation
except ValueError as error:
    print("refused:", error)

This is exactly how `Product` in the challenge protects its price (`src/supermarket/products.py`).

## 7 · A class that uses other classes: composition

A shopping cart is not a product, but it **has** products. A class that keeps objects of other classes in its attributes uses **composition**. Here the cart keeps a list.

For the rest of the notebook we use one complete `Product` class.

In [ ]:
class Product:
    """Something on the shelf, with a name and a net price (before VAT)."""

    def __init__(self, name: str, net_price: float) -> None:
        if net_price < 0:
            raise ValueError(f"the price of {name!r} cannot be negative")
        self.name = name
        self.net_price = net_price

    def vat_rate(self) -> float:
        """The VAT rate as a fraction: 7 % for now."""
        return 0.07

    def gross_price(self) -> float:
        """The shelf price in EUR: net price plus VAT, rounded to cents."""
        return round(self.net_price * (1 + self.vat_rate()), 2)

    def __repr__(self) -> str:
        return f"{type(self).__name__}({self.name!r}, {self.net_price})"


class ShoppingCart:
    """The items a customer has chosen, one entry per item."""

    def __init__(self) -> None:
        self._items: list[Product] = []          # a new cart is empty

    def add(self, product: Product, quantity: int = 1) -> None:
        """Put `quantity` items of a product in the cart."""
        if quantity < 1:
            raise ValueError("add at least one item")
        for _ in range(quantity):
            self._items.append(product)

    def item_count(self) -> int:
        """How many items are in the cart."""
        return len(self._items)

    def subtotal(self) -> float:
        """The sum of the shelf prices of all items, in EUR."""
        return round(sum(product.gross_price() for product in self._items), 2)

**Predict.** One bread and two milk: how many items, and what subtotal?

In [ ]:
bread = Product("Bread", 2.80)
milk = Product("Milk 1 l", 1.09)

cart = ShoppingCart()
cart.add(bread)
cart.add(milk, quantity=2)

print(cart.item_count())     # 3
print(cart.subtotal())       # 3.00 + 2 * 1.17 = 5.34

The cart does not compute prices itself: it asks each product with `product.gross_price()`. Each class does its own job. The challenge's `ShoppingCart` (`src/supermarket/cart.py`) works the same way.

> **Try it.** Add a method `is_empty()` to `ShoppingCart` that returns `True` when the cart has no items. Test it on a new cart and on `cart`.

In [ ]:
# your code here

## 8 · Inheritance: a new class from an existing one

Food has 7 % VAT, but dish soap has 19 %. We could add an `if` inside `vat_rate()`, but every new kind of product would make it longer. Instead we write **subclasses**: classes that take everything from `Product` (the **parent**) and change only what is different.

`class NonFoodProduct(Product):` reads "a non-food product **is a** product". The subclass **overrides** `vat_rate()`; everything else, `__init__`, `gross_price()` and `__repr__`, is inherited.

In [ ]:
class FoodProduct(Product):
    """Food: 7 % VAT."""

    def vat_rate(self) -> float:
        return 0.07


class NonFoodProduct(Product):
    """Everything that is neither food nor a drink, such as soap: 19 % VAT."""

    def vat_rate(self) -> float:
        return 0.19


bread = FoodProduct("Bread", 2.80)
dish_soap = NonFoodProduct("Dish soap", 1.95)

print(bread, bread.gross_price())           # 3.0
print(dish_soap, dish_soap.gross_price())   # 1.95 * 1.19 = 2.32

`gross_price()` is written only once, in `Product`. It calls `self.vat_rate()`, and Python uses the version of the object's own class. `isinstance()` checks the "is a" relation:

In [ ]:
print(isinstance(dish_soap, NonFoodProduct))   # True
print(isinstance(dish_soap, Product))          # True: a non-food product is a product
print(isinstance(dish_soap, FoodProduct))      # False
print(issubclass(NonFoodProduct, Product))     # True

### `super()`: extend the parent instead of copying it

A bottled drink has 19 % VAT **and** a deposit of 0.25 EUR per bottle. It needs one more attribute and a longer price rule. `super()` calls the parent's version of a method, so the subclass only adds what is new.

In [ ]:
class BottledDrink(Product):
    """A drink in a deposit bottle: 19 % VAT, plus the deposit."""

    def __init__(self, name: str, net_price: float, deposit: float = 0.25) -> None:
        super().__init__(name, net_price)    # the parent sets name and net_price and checks the price
        self.deposit = deposit               # only the new attribute is set here

    def vat_rate(self) -> float:
        return 0.19

    def gross_price(self) -> float:
        """The parent's shelf price plus the deposit."""
        return round(super().gross_price() + self.deposit, 2)


water = BottledDrink("Mineral water", 0.49)
print(water.gross_price())      # 0.49 * 1.19 = 0.58, plus 0.25: 0.83

**Predict.** Without `super().__init__(...)`, the object would have no `name` and no `net_price`. Why?

### Polymorphism: one call, the right behaviour for each object

The cart from part 7 works with all three subclasses without any change: it calls `gross_price()`, and each object answers in its own way. This is **polymorphism**.

In [ ]:
cart = ShoppingCart()
cart.add(FoodProduct("Bread", 2.80))
cart.add(FoodProduct("Milk 1 l", 1.09), quantity=2)
cart.add(BottledDrink("Mineral water", 0.49), quantity=6)
cart.add(NonFoodProduct("Dish soap", 1.95))

print(cart.item_count())     # 10 items
print(cart.subtotal())       # 3.00 + 2.34 + 4.98 + 2.32 = 12.64

These are the products of Ana's basket in the challenge, and 12.64 EUR is the subtotal that `uv run htw-checkout ana` prints.

### Polymorphism, step by step

Here is the same idea without the cart, so you can follow every call. We put three **different** kinds of product in one list and call the **same** method on each. We do not check the type anywhere: each object knows its own rule.

**Predict.** Which VAT rate will each line print?

In [ ]:
products = [
    FoodProduct("Bread", 2.80),            # a FoodProduct
    NonFoodProduct("Dish soap", 1.95),     # a NonFoodProduct
    BottledDrink("Mineral water", 0.49),   # a BottledDrink
]

for product in products:                   # the same two calls for every object
    rate = product.vat_rate()              # FoodProduct answers 0.07, the other two 0.19
    price = product.gross_price()          # BottledDrink adds its deposit, the others do not
    print(f"{type(product).__name__:<15} {product.name:<14} VAT {rate:.0%}  shelf price {price:.2f}")

What Python does for each call:

| Object | `vat_rate()` comes from | `gross_price()` comes from | Result |
|---|---|---|---|
| `FoodProduct("Bread", 2.80)` | `FoodProduct` | `Product` (inherited) | 2.80 × 1.07 = 3.00 |
| `NonFoodProduct("Dish soap", 1.95)` | `NonFoodProduct` | `Product` (inherited) | 1.95 × 1.19 = 2.32 |
| `BottledDrink("Mineral water", 0.49)` | `BottledDrink` | `BottledDrink`, which calls `Product`'s with `super()` | 0.58 + 0.25 = 0.83 |

The same is true for a function: `shelf_label` below accepts **any** product, because it only uses methods that every product has.

In [ ]:
def shelf_label(product: Product) -> str:
    """The text on the shelf label: name and shelf price. Works for every kind of product."""
    return f"{product.name}: {product.gross_price():.2f} EUR"


for product in products:
    print(shelf_label(product))

**Without polymorphism** the function would have to ask for the type and repeat each rule. Every new kind of product would mean one more `elif` here, and in every other place that computes a price:

In [ ]:
def shelf_price_without_polymorphism(product) -> float:
    """The same result, written with type checks: longer, and it must change for every new class."""
    if isinstance(product, BottledDrink):
        return round(product.net_price * 1.19 + product.deposit, 2)
    elif isinstance(product, NonFoodProduct):
        return round(product.net_price * 1.19, 2)
    elif isinstance(product, FoodProduct):
        return round(product.net_price * 1.07, 2)
    else:
        raise TypeError(f"unknown kind of product: {type(product).__name__}")


for product in products:
    print(product.name, shelf_price_without_polymorphism(product), product.gross_price())   # same numbers

> **Key point.** With polymorphism, adding a new kind of product means writing **one new class**; `shelf_label`, the cart and every loop keep working unchanged. Task 1 of the challenge (`FreshProduct`) works exactly like this.

### Abstract classes: a blueprint that must be completed

Should anyone create a plain `Product`? In the challenge, no: every product must say its VAT rate. An **abstract base class** (`ABC`) with an `@abstractmethod` makes that a rule: Python refuses to create objects of a class that has not written the abstract method.

In [ ]:
from abc import ABC, abstractmethod


class AbstractProduct(ABC):
    """A product that every subclass must complete with its VAT rate."""

    def __init__(self, name: str, net_price: float) -> None:
        self.name = name
        self.net_price = net_price

    @abstractmethod
    def vat_rate(self) -> float:
        """The VAT rate as a fraction. Every subclass must write this method."""

    def gross_price(self) -> float:
        return round(self.net_price * (1 + self.vat_rate()), 2)


class Tea(AbstractProduct):
    def vat_rate(self) -> float:
        return 0.07


print(Tea("Green tea", 2.00).gross_price())    # 2.14

try:
    AbstractProduct("Mystery", 1.00)
except TypeError as error:
    print("refused:", error)

In the challenge, `Product` and `Discount` are abstract in exactly this way (`products.py`, `discounts.py`).

> **Try it.** Write a subclass `Newspaper` of `Product` with 7 % VAT (printed press has the reduced rate in Germany) and create one with a net price of 2.50. What is its shelf price?

In [ ]:
# your code here

## From this notebook to the challenge

Everything you built here has a bigger version in the [supermarket challenge](../supermarket-challenge/README.md):

| Here | In the challenge |
|---|---|
| `Product` with a property for the price (part 6) | `Product` in `src/supermarket/products.py` |
| `FoodProduct`, `NonFoodProduct`, `BottledDrink` (part 8) | the same classes in `products.py` |
| `ShoppingCart` (part 7) | `ShoppingCart` in `cart.py` |
| `AbstractProduct` (part 8) | `Product` and `Discount`, both abstract |

```mermaid
classDiagram
    Product <|-- FoodProduct
    Product <|-- NonFoodProduct
    Product <|-- BottledDrink
    ShoppingCart o-- Product : holds
    class Product {
        name
        net_price
        vat_rate()
        gross_price()
    }
    class BottledDrink {
        deposit
        gross_price()
    }
```

Open the challenge's `products.py` now: you should recognise every line.

## Check your understanding

1. `bread = Product("Bread", 2.80)`. Who passes `self` to `__init__`?
   - A. You must write `Product(self, "Bread", 2.80)`
   - B. Python passes the new object automatically
   - C. `self` is a global variable
2. `cart.subtotal` without parentheses gives:
   - A. the subtotal
   - B. the method itself, not its result
   - C. an error
3. A class attribute `category = "food"` in `Product` is:
   - A. a separate value for each object
   - B. shared by all objects of the class
   - C. only visible inside `__init__`
4. `BottledDrink` writes `super().__init__(name, net_price)`. What happens if you delete that line?
   - A. Nothing changes
   - B. The object has no `name` and no `net_price`, so `gross_price()` fails
   - C. Python adds the line automatically

## Summary

| You want to | Write |
|---|---|
| define a blueprint | `class Product:` |
| give each object its data | `def __init__(self, name, net_price): self.name = name` |
| create an object | `bread = Product("Bread", 2.80)` |
| add behaviour | `def gross_price(self): ...`, called as `bread.gross_price()` |
| control how it prints | `__repr__` (developers), `__str__` (people) |
| document it | a docstring under `class` and `def`, type hints on parameters |
| refuse wrong values | `raise ValueError(...)`, a `@property` with a setter |
| use other objects | composition: keep them in an attribute, `self._items = []` |
| reuse and change a class | inheritance: `class FoodProduct(Product):`, override a method, `super()` |
| force subclasses to complete it | `class Product(ABC):` with `@abstractmethod` |

**Next:** the [supermarket challenge](../supermarket-challenge/README.md), and the lecture notebook [01-lecture-classes-errors-and-tests.ipynb](01-lecture-classes-errors-and-tests.ipynb) for errors and tests in depth.